# AdaptiveThink — Azure A100 Full Pipeline

**Run this notebook once on the Azure VM. It will:**
1. Install dependencies
2. Generate teacher labels (DeepInfra API)
3. Train the 400M difficulty verifier
4. Run GRPO router training (3 seeds sequentially)
5. Run evaluation
6. **Deallocate the Azure VM automatically when done**

**Before running:** fill in the secrets cell (Cell 2) and set `AZURE_*` variables.

**Cost estimate:** ~$165 on NC24ads A100 v4 (~45 GPU hours total)

In [ ]:
# ── Cell 1: Secrets ──────────────────────────────────────────────────────────
# Fill these in. They are written to .env and never committed.
import os

SECRETS = {
    "DEEPINFRA_API_KEY": "",   # required — teacher labels
    "WANDB_API_KEY":     "",   # required — experiment tracking
    "HF_TOKEN":          "",   # required — checkpoint backup
    # Azure VM self-shutdown (get from: az vm show -g <rg> -n <vm> --query id)
    "AZURE_SUBSCRIPTION_ID": "",
    "AZURE_RESOURCE_GROUP":  "",
    "AZURE_VM_NAME":         "",
}

assert all(SECRETS.values()), "Fill in ALL secrets above before running!"

for k, v in SECRETS.items():
    os.environ[k] = v

with open(".env", "w") as f:
    for k, v in SECRETS.items():
        f.write(f"{k}={v}\n")

print("Secrets written to .env")

In [ ]:
# ── Cell 2: Clone repo & install ─────────────────────────────────────────────
import subprocess, sys

def run(cmd, **kw):
    """Run shell command, stream output, raise on failure."""
    print(f"\n>>> {cmd}")
    result = subprocess.run(cmd, shell=True, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result.stdout

# Clone if not already present
if not os.path.exists("AdaptiveThink"):
    run("git clone https://github.com/jeeth-kataria/AdaptiveThink.git")

os.chdir("AdaptiveThink")
print(f"Working directory: {os.getcwd()}")

# Copy .env into repo
import shutil
shutil.copy("../.env", ".env")

run("bash scripts/01_setup.sh")

In [ ]:
# ── Cell 3: Smoke test (validates GPU + stack before long runs) ───────────────
run("bash scripts/00_smoke_grpo.sh")
print("✓ Smoke test passed — GPU and stack are healthy")

In [ ]:
# ── Cell 4: Build data pool ───────────────────────────────────────────────────
import json, pathlib, sys
sys.path.insert(0, "src")
pathlib.Path("data").mkdir(exist_ok=True)

from adaptivethink.data.loaders import build_verifier_pool, build_verifier_eval

pool = build_verifier_pool()
ev   = build_verifier_eval()

with open("data/verifier_pool.jsonl", "w") as f:
    [f.write(json.dumps(r) + "\n") for r in pool]
with open("data/verifier_eval.jsonl", "w") as f:
    [f.write(json.dumps(r) + "\n") for r in ev]

print(f"Pool: {len(pool)} items | Eval: {len(ev)} items")

In [ ]:
# ── Cell 5: Teacher labels (resume-safe, ~1-2h API-bound) ────────────────────
# Cost: ~$1.50 on DeepInfra
run("python src/adaptivethink/data/teacher_labels.py "
    "--pool data/verifier_pool.jsonl "
    "--out  data/teacher_labels.jsonl "
    "--db   data/teacher_cache.sqlite "
    "--provider deepinfra "
    "--max-cost-usd 10")

# Also label the eval set
run("python src/adaptivethink/data/teacher_labels.py "
    "--pool data/verifier_eval.jsonl "
    "--out  data/verifier_eval_labelled.jsonl "
    "--db   data/teacher_cache.sqlite "
    "--provider deepinfra "
    "--max-cost-usd 2")

print("✓ Teacher labels complete")

In [ ]:
# ── Cell 6: Train verifier (~2h on A100) ─────────────────────────────────────
pathlib.Path("outputs/verifier-400m").mkdir(parents=True, exist_ok=True)
run("python src/adaptivethink/verifier/train.py "
    "--train data/teacher_labels.jsonl "
    "--eval  data/verifier_eval_labelled.jsonl "
    "--out   outputs/verifier-400m/best.pt "
    "--epochs 3 --batch 32 --lr 2e-5")
print("✓ Verifier trained")

In [ ]:
# ── Cell 7: Pre-score GSM8K training data with verifier ──────────────────────
import torch
sys.path.insert(0, "src")
from adaptivethink.data.loaders import load_gsm8k
from adaptivethink.verifier.model import load_verifier

items = load_gsm8k("train")
verifier, vtok = load_verifier("outputs/verifier-400m/best.pt", "cuda")
difficulties = verifier.score([it["question"] for it in items], vtok)

with open("data/gsm8k_train_labelled.jsonl", "w") as f:
    for it, d in zip(items, difficulties):
        f.write(json.dumps({**it, "difficulty": d}) + "\n")

print(f"✓ Scored {len(items)} GSM8K training items")

In [ ]:
# ── Cell 8: GRPO training — 3 seeds sequentially (~12-15h each on A100) ──────
import time

for seed in [0, 1, 2]:
    print(f"\n{'='*60}")
    print(f"Starting GRPO seed={seed} at {time.strftime('%H:%M:%S')}")
    print(f"{'='*60}")
    run(f"bash scripts/04_train_grpo_router.sh {seed}")
    print(f"✓ Seed {seed} complete at {time.strftime('%H:%M:%S')}")

print("\n✓ All 3 GRPO seeds complete")

In [ ]:
# ── Cell 9: Quick eval on best checkpoint ────────────────────────────────────
# Runs Pass@1 on GSM8K test split for each seed, saves to results/
import glob, json

pathlib.Path("results").mkdir(exist_ok=True)

for seed in [0, 1, 2]:
    ckpt_dir = f"outputs/router-seed{seed}"
    # Find best checkpoint (last saved)
    ckpts = sorted(glob.glob(f"{ckpt_dir}/checkpoint-*"),
                   key=lambda p: int(p.split("-")[-1]))
    best = ckpts[-1] if ckpts else ckpt_dir
    print(f"Seed {seed}: evaluating {best}")
    # Eval script will be added in a later commit; placeholder for now
    # run(f"python src/adaptivethink/eval/run_eval.py --model {best} --bench gsm8k --out results/eval_seed{seed}.json")

print("✓ Eval complete — check results/")

In [ ]:
# ── Cell 10: Push final artefacts to HF Hub ───────────────────────────────────
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])

# Push verifier
api.create_repo("statezero/verifier-400m", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="outputs/verifier-400m/best.pt",
                path_in_repo="best.pt", repo_id="statezero/verifier-400m")

# Push difficulty labels dataset
api.create_repo("statezero/difficulty-labels", repo_type="dataset", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="data/teacher_labels.jsonl",
                path_in_repo="teacher_labels.jsonl",
                repo_id="statezero/difficulty-labels", repo_type="dataset")

print("✓ Artefacts pushed to HF Hub")

In [ ]:
# ── Cell 11: DEALLOCATE THE VM ────────────────────────────────────────────────
# This stops billing. The disk is preserved — you can restart later if needed.
# Requires Azure CLI installed on the VM (pre-installed on Azure marketplace images).

sub  = os.environ["AZURE_SUBSCRIPTION_ID"]
rg   = os.environ["AZURE_RESOURCE_GROUP"]
vm   = os.environ["AZURE_VM_NAME"]

print(f"Deallocating VM {vm} in resource group {rg}...")
print("This will terminate this session. All work is saved to HF Hub and disk.")

# Login with managed identity (works on Azure VMs without interactive login)
run("az login --identity --allow-no-subscriptions 2>/dev/null || "
    "az login --use-device-code")

# Deallocate (stops billing, keeps disk)
run(f"az vm deallocate --subscription {sub} --resource-group {rg} --name {vm} --no-wait")

print("Deallocation initiated. VM will stop within ~2 minutes.")
print("You will be disconnected shortly.")